# NTT PEEL Artifact Demo

![Alt text](Artifacts.PNG)

This notebook serves as the main orchestration notebook and calls all submodules of the attack pipeline. The complete attack consists of four stages. Each stage is optional, as we provide pre-generated outputs that allow users to start from later steps if desired.

* Step 1: Generate In-Place Modular Reduction Flags
In this step, we simulate small polynomials 𝑓 and 𝑔 using the same Gaussian distribution as in FALCON. The simulation produces in-place modular reduction flags. This step is included because real power measurements may not be available in the user’s environment.

* Step 2: Run Forward Inference
This stage performs forward inference using the reduction flags. For convenience, we provide a set of pre-recorded flags so that users may begin directly from this step.
This stage requires: A CUDA-compatible GPU, At least 1.5 TB of disk space and up to two weeks of runtime per coefficient attack, depending on hardware

* Step 3: Run Backward Inference
This step executes the backward inference process and generates the pruned keyset values. Since the inference typically takes around two weeks. To reduce the burden on users, we also provide a completed output of the Guess-and-Prune (G&P) process.

* Step 4: Perform Lattice-Reduction-Based Pruning
In the final stage, we apply lattice-reduction-based pruning and selection to the output file Keyset_Values.csv. This step reports the remaining unknown coefficients and their locations.
Since Steps 2 and 3 are computationally expensive, we also include a finalized keyset file for users who wish to start directly from this stage.

**Contents of the repository**
* This repository provides the core implementation of the attack framework described in the paper, including:
1. Guess-and-Prune attack logic
2. Lattice-reduction-based pruning
3. Tools for simulating modular reduction flags

* The following components are not included:
1. Power measurement traces and tooling
2. Methods for extracting Points of Interest (POIs). These components are documented in previous publications.

## Import libraries to be Used

In [1]:
import csv
import random
from sympy import mod_inverse

import os
import itertools
import pandas as pd
import numpy as np
from sympy import mod_inverse
from concurrent.futures import ThreadPoolExecutor, as_completed
import re

import subprocess
import sys
from pathlib import Path


# ---- Require CuPy/CUDA (exit if unavailable) ----
try:
    import cupy as cp
    from cupy.cuda.runtime import getDeviceCount
    # Ensure at least one CUDA device is present
    if getDeviceCount() < 1:
        raise RuntimeError("No CUDA-capable device found.")
except Exception as e:
    # Fail fast with a clear error message
    raise SystemExit(
        "ERROR: CuPy/CUDA is required for this program but is not available.\n"
        f"Details: {e}"
    )

import argparse, ast, math, itertools
import numpy as np
import pandas as pd

# --------------------------------
# Optional fpylll backend
# --------------------------------
FPYLLL_OK, FPY_ERR = False, None
try:
    from fpylll import IntegerMatrix, LLL as FLLL, BKZ, GSO
    FPYLLL_OK = True
except Exception as e:
    FPY_ERR = e


Global Parameters

In [2]:
# -----------------------------
# Constants and global settings
# -----------------------------
q = 12289
Q = 12289
R = 2**16
Rinv = mod_inverse(R, q)
# Parameters
N = 512
logN = 9
GMb_mont = [
	 4091,  7888, 11060, 11208,  6960,  4342,  6275,  9759,
	 1591,  6399,  9477,  5266,   586,  5825,  7538,  9710,
	 1134,  6407,  1711,   965,  7099,  7674,  3743,  6442,
	10414,  8100,  1885,  1688,  1364, 10329, 10164,  9180,
	12210,  6240,   997,   117,  4783,  4407,  1549,  7072,
	 2829,  6458,  4431,  8877,  7144,  2564,  5664,  4042,
	12189,   432, 10751,  1237,  7610,  1534,  3983,  7863,
	 2181,  6308,  8720,  6570,  4843,  1690,    14,  3872,
	 5569,  9368, 12163,  2019,  7543,  2315,  4673,  7340,
	 1553,  1156,  8401, 11389,  1020,  2967, 10772,  7045,
	 3316, 11236,  5285, 11578, 10637, 10086,  9493,  6180,
	 9277,  6130,  3323,   883, 10469,   489,  1502,  2851,
	11061,  9729,  2742, 12241,  4970, 10481, 10078,  1195,
	  730,  1762,  3854,  2030,  5892, 10922,  9020,  5274,
	 9179,  3604,  3782, 10206,  3180,  3467,  4668,  2446,
	 7613,  9386,   834,  7703,  6836,  3403,  5351, 12276,
	 3580,  1739, 10820,  9787, 10209,  4070, 12250,  8525,
	10401,  2749,  7338, 10574,  6040,   943,  9330,  1477,
	 6865,  9668,  3585,  6633, 12145,  4063,  3684,  7680,
	 8188,  6902,  3533,  9807,  6090,   727, 10099,  7003,
	 6945,  1949,  9731, 10559,  6057,   378,  7871,  8763,
	 8901,  9229,  8846,  4551,  9589, 11664,  7630,  8821,
	 5680,  4956,  6251,  8388, 10156,  8723,  2341,  3159,
	 1467,  5460,  8553,  7783,  2649,  2320,  9036,  6188,
	  737,  3698,  4699,  5753,  9046,  3687,    16,   914,
	 5186, 10531,  4552,  1964,  3509,  8436,  7516,  5381,
	10733,  3281,  7037,  1060,  2895,  7156,  8887,  5357,
	 6409,  8197,  2962,  6375,  5064,  6634,  5625,   278,
	  932, 10229,  8927,  7642,   351,  9298,   237,  5858,
	 7692,  3146, 12126,  7586,  2053, 11285,  3802,  5204,
	 4602,  1748, 11300,   340,  3711,  4614,   300, 10993,
	 5070, 10049, 11616, 12247,  7421, 10707,  5746,  5654,
	 3835,  5553,  1224,  8476,  9237,  3845,   250, 11209,
	 4225,  6326,  9680, 12254,  4136,  2778,   692,  8808,
	 6410,  6718, 10105, 10418,  3759,  7356, 11361,  8433,
	 6437,  3652,  6342,  8978,  5391,  2272,  6476,  7416,
	 8418, 10824, 11986,  5733,   876,  7030,  2167,  2436,
	 3442,  9217,  8206,  4858,  5964,  2746,  7178,  1434,
	 7389,  8879, 10661, 11457,  4220,  1432, 10832,  4328,
	 8557,  1867,  9454,  2416,  3816,  9076,   686,  5393,
	 2523,  4339,  6115,   619,   937,  2834,  7775,  3279,
	 2363,  7488,  6112,  5056,   824, 10204, 11690,  1113,
	 2727,  9848,   896,  2028,  5075,  2654, 10464,  7884,
	12169,  5434,  3070,  6400,  9132, 11672, 12153,  4520,
	 1273,  9739, 11468,  9937, 10039,  9720,  2262,  9399,
	11192,   315,  4511,  1158,  6061,  6751, 11865,   357,
	 7367,  4550,   983,  8534,  8352, 10126,  7530,  9253,
	 4367,  5221,  3999,  8777,  3161,  6990,  4130, 11652,
	 3374, 11477,  1753,   292,  8681,  2806, 10378, 12188,
	 5800, 11811,  3181,  1988,  1024,  9340,  2477, 10928,
	 4582,  6750,  3619,  5503,  5233,  2463,  8470,  7650,
	 7964,  6395,  1071,  1272,  3474, 11045,  3291, 11344,
	 8502,  9478,  9837,  1253,  1857,  6233,  4720, 11561,
	 6034,  9817,  3339,  1797,  2879,  6242,  5200,  2114,
	 7962,  9353, 11363,  5475,  6084,  9601,  4108,  7323,
	10438,  9471,  1271,   408,  6911,  3079,   360,  8276,
	11535,  9156,  9049, 11539,   850,  8617,   784,  7919,
	 8334, 12170,  1846, 10213, 12184,  7827, 11903,  5600,
	 9779,  1012,   721,  2784,  6676,  6552,  5348,  4424,
	 6816,  8405,  9959,  5150,  2356,  5552,  5267,  1333,
	 8801,  9661,  7308,  5788,  4910,   909, 11613,  4395,
	 8238,  6686,  4302,  3044,  2285, 12249,  1963,  9216,
	 4296, 11918,   695,  4371,  9793,  4884,  2411, 10230,
	 2650,   841,  3890, 10231,  7248,  8505, 11196,  6688,
	 4059,  6060,  3686,  4722, 11853,  5816,  7058,  6868,
	11137,  7926,  4894, 12284,  4102,  3908,  3610,  6525,
	 7938,  7982, 11977,  6755,   537,  4562,  1623,  8227,
	11453,  7544,   906, 11816,  9548, 10858,  9703,  2815,
	11736,  6813,  6979,   819,  8903,  6271, 10843,   348,
	 7514,  8339,  6439,   694,   852,  5659,  2781,  3716,
	11589,  3024,  1523,  8659,  4114, 10738,  3303,  5885,
	 2978,  7289, 11884,  9123,  9323, 11830,    98,  2526,
	 2116,  4131, 11407,  1844,  3645,  3916,  8133,  2224,
	10871,  8092,  9651,  5989,  7140,  8480,  1670,   159,
	10923,  4918,   128,  7312,   725,  9157,  5006,  6393,
	 3494,  6043, 10972,  6181, 11838,  3423, 10514,  7668,
	 3693,  6658,  6905, 11953, 10212, 11922,  9101,  8365,
	 5110,    45,  2400,  1921,  4377,  2720,  1695,    51,
	 2808,   650,  1896,  9997,  9971, 11980,  8098,  4833,
	 4135,  4257,  5838,  4765, 10985, 11532,   590, 12198,
	  482, 12173,  2006,  7064, 10018,  3912, 12016, 10519,
	11362,  6954,  2210,   284,  5413,  6601,  3865, 10339,
	11188,  6231,   517,  9564, 11281,  3863,  1210,  4604,
	 8160, 11447,   153,  7204,  5763,  5089,  9248, 12154,
	11748,  1354,  6672,   179,  5532,  2646,  5941, 12185,
	  862,  3158,   477,  7279,  5678,  7914,  4254,   302,
	 2893, 10114,  6890,  9560,  9647, 11905,  4098,  9824,
	10269,  1353, 10715,  5325,  6254,  3951,  1807,  6449,
	 5159,  1308,  8315,  3404,  1877,  1231,   112,  6398,
	11724, 12272,  7286,  1459, 12274,  9896,  3456,   800,
	 1397, 10678,   103,  7420,  7976,   936,   764,   632,
	 7996,  8223,  8445,  7758, 10870,  9571,  2508,  1946,
	 6524, 10158,  1044,  4338,  2457,  3641,  1659,  4139,
	 4688,  9733, 11148,  3946,  2082,  5261,  2036, 11850,
	 7636, 12236,  5366,  2380,  1399,  7720,  2100,  3217,
	10912,  8898,  7578, 11995,  2791,  1215,  3355,  2711,
	 2267,  2004,  8568, 10176,  3214,  2337,  1750,  4729,
	 4997,  7415,  6315, 12044,  4374,  7157,  4844,   211,
	 8003, 10159,  9290, 11481,  1735,  2336,  5793,  9875,
	 8192,   986,  7527,  1401,   870,  3615,  8465,  2756,
	 9770,  2034, 10168,  3264,  6132,    54,  2880,  4763,
	11805,  3074,  8286,  9428,  4881,  6933,  1090, 10038,
	 2567,   708,   893,  6465,  4962, 10024,  2090,  5718,
	10743,   780,  4733,  4623,  2134,  2087,  4802,   884,
	 5372,  5795,  5938,  4333,  6559,  7549,  5269, 10664,
	 4252,  3260,  5917, 10814,  5768,  9983,  8096,  7791,
	 6800,  7491,  6272,  1907, 10947,  6289, 11803,  6032,
	11449,  1171,  9201,  7933,  2479,  7970, 11337,  7062,
	 8911,  6728,  6542,  8114,  8828,  6595,  3545,  4348,
	 4610,  2205,  6999,  8106,  5560, 10390,  9321,  2499,
	 2413,  7272,  6881, 10582,  9308,  9437,  3554,  3326,
	 5991, 11969,  3415, 12283,  9838, 12063,  4332,  7830,
	11329,  6605, 12271,  2044, 11611,  7353, 11201, 11582,
	 3733,  8943,  9978,  1627,  7168,  3935,  5050,  2762,
	 7496, 10383,   755,  1654, 12053,  4952, 10134,  4394,
	 6592,  7898,  7497,  8904, 12029,  3581, 10748,  5674,
	10358,  4901,  7414,  8771,   710,  6764,  8462,  7193,
	 5371,  7274, 11084,   290,  7864,  6827, 11822,  2509,
	 6578,  4026,  5807,  1458,  5721,  5762,  4178,  2105,
	11621,  4852,  8897,  2856, 11510,  9264,  2520,  8776,
	 7011,  2647,  1898,  7039,  5950, 11163,  5488,  6277,
	 9182, 11456,   633, 10046, 11554,  5633,  9587,  2333,
	 7008,  7084,  5047,  7199,  9865,  8997,   569,  6390,
	10845,  9679,  8268, 11472,  4203,  1997,     2,  9331,
	  162,  6182,  2000,  3649,  9792,  6363,  7557,  6187,
	 8510,  9935,  5536,  9019,  3706, 12009,  1452,  3067,
	 5494,  9692,  4865,  6019,  7106,  9610,  4588, 10165,
	 6261,  5887,  2652, 10172,  1580, 10379,  4638,  9949,
]  # ← paste full GMb array from your message
# Convert to normal domain
GMb_normal = [(tw * Rinv) % Q for tw in GMb_mont]


Helper Functions

In [3]:
def sample_falcon_coeff():
    # Draw from Gaussian with mean 0, stddev 1.433
    val = int(round(random.gauss(0, 1.433)))
    if val < 0:
        # Map to ring counterpart
        return Q + val
    return val

def mq_add(x, y):
    d = x + y - Q
    if d < 0:
        d += Q
    return d

def mq_sub(x, y):
    d = x - y
    if d < 0:
        d += Q
    return d

def mq_NTT(a, logn, GMb_normal, csv_writer):
    n = 1 << logn
    t = n
    stage = 0
    m = 1
    gmb_idx = 1  # skip GMb[0] = 4091 (usually not used for twiddles)

    while m < n:
        ht = t >> 1
        butterfly_counter = 0
        group_count = m

        for group in range(group_count):
            j1 = group * t
            s_norm = GMb_normal[gmb_idx]
            gmb_idx += 1

            for j in range(j1, j1 + ht):
                u = a[j]
                x = a[j + ht]

                raw_tw = x * s_norm
                v = raw_tw % Q
                tw_norm = int(raw_tw != v)

                raw_add = u + v
                a1p = mq_add(u, v)
                add_norm = int(raw_add != a1p)

                raw_sub = u - v
                a2p = mq_sub(u, v)
                sub_norm = int(raw_sub != a2p)

                a[j] = a1p
                a[j + ht] = a2p

                csv_writer.writerow([stage, butterfly_counter, tw_norm, add_norm, sub_norm])
                butterfly_counter += 1

        t = ht
        stage += 1
        m <<= 1

Step 1: Generate the reduction flags (Gaussian Distribution)

In [4]:
def Victim():
    poly = [sample_falcon_coeff() for _ in range(N)]

    with open("C:/Users/jqiu2/Desktop/Artifact_Demo/reduc_flags_GAUSS.csv", mode='w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["stage", "butterfly_idx", "twiddle_norm", "add_norm", "sub_norm"])
        mq_NTT(poly, logN, GMb_normal, writer)

    print("Final NTT output (first 16 values):", poly[:16])
    print("NTT complete. Normalization flags written to reduc_flags_GAUSS.csv")

Victim()

Final NTT output (first 16 values): [567, 10860, 1424, 11567, 7066, 12128, 3323, 3980, 3474, 2160, 9471, 6964, 5407, 11653, 4125, 8798]
NTT complete. Normalization flags written to reduc_flags_GAUSS.csv


(Optional) Step 2:  perform Guess and Prune-Forward(A GPU Compatible device is required, it takes around 14 days to complete an attack on a polynomial.) We included our run results of steps 2 and 3 to make the lattice attack execution possible without running steps 2 and 3. 

In [5]:
#Call and execute the G&P 
cmd_f = [
    sys.executable,  # uses the same python interpreter you are currently running
    "C:/Users/jqiu2/Desktop/Artifact_Demo/NTT_Forward_GPU_Range_Upload.py"]

p = subprocess.Popen(
    cmd_f,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # merge stderr into stdout
    text=True,                 # decode to str
    bufsize=1,                 # line-buffered (best-effort)
)

for line in p.stdout:
    print(line, end="")        # real-time in notebook output

rc = p.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd_f)

[S0] Butterfly 0 done.
[S0] Butterfly 1 done.
[S0] Butterfly 2 done.
[S0] Butterfly 3 done.
[S0] Butterfly 4 done.
[S0] Butterfly 5 done.
[S0] Butterfly 6 done.
[S0] Butterfly 7 done.
[S0] Butterfly 8 done.
[S0] Butterfly 9 done.
[S0] Butterfly 10 done.
[S0] Butterfly 11 done.
[S0] Butterfly 12 done.
[S0] Butterfly 13 done.
[S0] Butterfly 14 done.
[S0] Butterfly 15 done.
[S0] Butterfly 16 done.
[S0] Butterfly 17 done.
[S0] Butterfly 18 done.
[S0] Butterfly 19 done.
[S0] Butterfly 20 done.
[S0] Butterfly 21 done.
[S0] Butterfly 22 done.
[S0] Butterfly 23 done.
[S0] Butterfly 24 done.
[S0] Butterfly 25 done.
[S0] Butterfly 26 done.
[S0] Butterfly 27 done.
[S0] Butterfly 28 done.
[S0] Butterfly 29 done.
[S0] Butterfly 30 done.
[S0] Butterfly 31 done.
[S0] Butterfly 32 done.
[S0] Butterfly 33 done.
[S0] Butterfly 34 done.
[S0] Butterfly 35 done.
[S0] Butterfly 36 done.
[S0] Butterfly 37 done.
[S0] Butterfly 38 done.
[S0] Butterfly 39 done.
[S0] Butterfly 40 done.
[S0] Butterfly 41 done.
[S

KeyboardInterrupt: 

(Optional) Step 3:  perform Guess and Prune-Backward. We included our run results of steps 2 and 3 to make the lattice attack execution possible without running steps 2 and 3. Therefore, please feel free to interrupt(terminate) the running instance of steps 2 and 3, with peace in mind that the following steps will not be impacted.

In [6]:
cmd_b = [
    sys.executable,  # uses the same python interpreter you are currently running
    "C:/Users/jqiu2/Desktop/Artifact_Demo/NTT_Backward_Upload.py"]

p = subprocess.Popen(
    cmd_b,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # merge stderr into stdout
    text=True,                 # decode to str
    bufsize=1,                 # line-buffered (best-effort)
)

for line in p.stdout:
    print(line, end="")        # real-time in notebook output

rc = p.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd_b)


=== Backward Tracing stage0_butterfly0 ===
filtering completed

=== Backward Tracing stage0_butterfly1 ===
filtering completed

=== Backward Tracing stage0_butterfly2 ===
filtering completed

=== Backward Tracing stage0_butterfly3 ===
filtering completed

=== Backward Tracing stage0_butterfly4 ===
filtering completed

=== Backward Tracing stage0_butterfly5 ===
filtering completed

=== Backward Tracing stage0_butterfly6 ===
filtering completed

=== Backward Tracing stage0_butterfly7 ===
filtering completed

=== Backward Tracing stage0_butterfly8 ===
filtering completed

=== Backward Tracing stage0_butterfly9 ===
filtering completed

=== Backward Tracing stage0_butterfly10 ===
filtering completed

=== Backward Tracing stage0_butterfly11 ===
filtering completed

=== Backward Tracing stage0_butterfly12 ===
filtering completed

=== Backward Tracing stage0_butterfly13 ===
filtering completed

=== Backward Tracing stage0_butterfly14 ===
filtering completed

=== Backward Tracing stage0_butter

Perform Lattice reduction based pruning. This takes around 25 minutes on an Intel 1068-NG7 CPU

In [7]:
# Windows path -> WSL path
WSL_WORKDIR = "/mnt/c/Users/jqiu2/Desktop/Artifact_Demo"

bash_cmd = (
    "source ~/miniforge3/etc/profile.d/conda.sh && "
    "conda activate latt && "
    f"cd {WSL_WORKDIR} && "
    "python -u Lattice_Reduction.py KeySet_Values.csv "
    "--prefer_sympy --block 40 --tscale 1000000 --rounds 3"
)

cmd = ["wsl", "-e", "bash", "-lc", bash_cmd]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
if rc != 0:
    raise RuntimeError(f"WSL lattice run failed (exit code {rc})")


=== Round 1/3 ===
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[*] LLL reduction (mpfr)...
[*] BKZ reduction (mpfr, block size = 40) ...
[*] Reduction complete.
[